In [1]:
import sys

print("Python:", sys.version)
print("Ambiente:", sys.executable)

Python: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
Ambiente: c:\Users\Dell\Documents\rocketseat\ai-customer-intelligence\.venv\Scripts\python.exe


In [4]:
import pandas as pd

print("Versão do Pandas:", pd.__version__)

Versão do Pandas: 3.1.1


In [3]:
%pip install pandas

  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   --------- ------------------------------ 2.6/10.9 MB 24.4 MB/s eta 0:00:01
   ---------------------------------------  10.7/10.9 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 32.9 MB/s  0:00:00
Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl (12.7 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   -


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path

caminho = Path("../data/raw/orders.csv")

pedidos = pd.read_csv(caminho)

print("Linhas:", pedidos.shape[0])
print("Colunas:", pedidos.shape[1])

pedidos.head()

Linhas: 368999
Colunas: 29


,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,...,order_moment_delivering,order_moment_delivered,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time
0,68405119,3512,5,68405119,68405119,CANCELED,62.7,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,68405123,3512,5,68405123,68405123,CANCELED,62.7,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68405206,3512,5,68405206,68405206,CANCELED,115.5,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,68405465,3401,5,68405465,68405465,CANCELED,55.9,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,68406064,3401,5,68406064,68406064,CANCELED,37.9,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
pedidos.info()

<class 'pandas.DataFrame'>
RangeIndex: 368999 entries, 0 to 368998
Data columns (total 29 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   order_id                           368999 non-null  int64  
 1   store_id                           368999 non-null  int64  
 2   channel_id                         368999 non-null  int64  
 3   payment_order_id                   368999 non-null  int64  
 4   delivery_order_id                  368999 non-null  int64  
 5   order_status                       368999 non-null  str    
 6   order_amount                       368999 non-null  float64
 7   order_delivery_fee                 368999 non-null  float64
 8   order_delivery_cost                361794 non-null  float64
 9   order_created_hour                 368999 non-null  int64  
 10  order_created_minute               368999 non-null  int64  
 11  order_created_day                  368999 non-null

In [7]:
pedidos["order_status"].value_counts()

order_status
FINISHED    352020
CANCELED     16979
Name: count, dtype: int64

In [8]:
total_pedidos = len(pedidos)

pedidos_cancelados = (
    pedidos["order_status"] == "CANCELED"
).sum()

taxa_cancelamento = (
    pedidos_cancelados / total_pedidos
) * 100

print(f"Total de pedidos: {total_pedidos:,}")
print(f"Pedidos cancelados: {pedidos_cancelados:,}")
print(f"Taxa de cancelamento: {taxa_cancelamento:.2f}%")

Total de pedidos: 368,999
Pedidos cancelados: 16,979
Taxa de cancelamento: 4.60%


In [9]:
pedidos.groupby("order_status")[
    ["order_moment_delivered", "order_metric_cycle_time"]
].count()

,order_moment_delivered,order_metric_cycle_time
order_status,,
CANCELED,598,1723
FINISHED,19003,351657


In [10]:
colunas_tempo = [
    "order_moment_created",
    "order_moment_accepted",
    "order_moment_ready",
    "order_moment_collected",
    "order_moment_in_expedition",
    "order_moment_delivering",
    "order_moment_delivered",
    "order_moment_finished"
]

pedidos.groupby("order_status")[colunas_tempo].count().T

order_status,CANCELED,FINISHED
order_moment_created,16979,352020
order_moment_accepted,8886,350652
order_moment_ready,2916,340977
order_moment_collected,1849,324256
order_moment_in_expedition,1671,299899
order_moment_delivering,1317,342366
order_moment_delivered,598,19003
order_moment_finished,1383,352017


In [11]:
pedidos_finalizados = pedidos[
    pedidos["order_status"] == "FINISHED"
]

pedidos_finalizados["order_metric_cycle_time"].describe()

count    351657.000000
mean        155.843596
std        1225.229132
min           0.370000
25%          32.870000
50%          42.180000
75%          55.550000
max      154277.320000
Name: order_metric_cycle_time, dtype: float64

In [12]:
tempos = pedidos_finalizados["order_metric_cycle_time"]

q1 = tempos.quantile(0.25)
q3 = tempos.quantile(0.75)

iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = tempos[
    (tempos < limite_inferior) |
    (tempos > limite_superior)
]

print(f"Limite inferior: {limite_inferior:.2f}")
print(f"Limite superior: {limite_superior:.2f}")
print(f"Quantidade de outliers: {len(outliers):,}")
print(f"Percentual de outliers: {len(outliers) / tempos.count() * 100:.2f}%")

Limite inferior: -1.15
Limite superior: 89.57
Quantidade de outliers: 31,635
Percentual de outliers: 9.00%


In [13]:
tempos_sem_outliers = tempos[
    (tempos >= limite_inferior) &
    (tempos <= limite_superior)
]

print(f"Média original: {tempos.mean():.2f}")
print(f"Mediana original: {tempos.median():.2f}")

print(f"Média sem outliers: {tempos_sem_outliers.mean():.2f}")
print(f"Mediana sem outliers: {tempos_sem_outliers.median():.2f}")

Média original: 155.84
Mediana original: 42.18
Média sem outliers: 42.45
Mediana sem outliers: 40.47


### Análise dos valores atípicos no tempo de ciclo

Foram analisados 351.657 registros válidos de pedidos finalizados.

O método IQR identificou 31.635 valores atípicos (9,00%), com limite superior de 89,57.

A média original foi de 155,84, enquanto a mediana foi de 42,18. Após a exclusão temporária dos outliers, a média caiu para 42,45 e a mediana para 40,47.

Essa diferença indica forte influência dos valores extremos sobre a média.

**Decisão analítica:** preservar os registros originais, utilizar a mediana como referência do comportamento típico e investigar separadamente os valores extremos.

A unidade de medida e o significado operacional do tempo de ciclo ainda precisam ser confirmados.


In [14]:
tempos = pedidos_finalizados["order_metric_cycle_time"]

print("Tempos negativos:", (tempos < 0).sum())
print("Tempos zerados:", (tempos == 0).sum())

print("\nValores acima dos limites:")

for limite in [60, 120, 240, 1440]:
    quantidade = (tempos > limite).sum()
    percentual = quantidade / tempos.count() * 100

    print(
        f"Acima de {limite}: "
        f"{quantidade:,} ({percentual:.2f}%)"
    )

Tempos negativos: 0
Tempos zerados: 0

Valores acima dos limites:
Acima de 60: 71,150 (20.23%)
Acima de 120: 24,299 (6.91%)
Acima de 240: 17,281 (4.91%)
Acima de 1440: 7,469 (2.12%)


In [15]:
amostra = pedidos_finalizados[
    [
        "order_id",
        "order_moment_created",
        "order_moment_finished",
        "order_metric_cycle_time"
    ]
].dropna().head(10).copy()

inicio = pd.to_datetime(amostra["order_moment_created"])
fim = pd.to_datetime(amostra["order_moment_finished"])

amostra["diferenca_minutos"] = (
    (fim - inicio).dt.total_seconds() / 60
)

amostra["diferenca_horas"] = (
    amostra["diferenca_minutos"] / 60
)

amostra

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\2654485319.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fim = pd.to_datetime(amostra["order_moment_finished"])


,order_id,order_moment_created,order_moment_finished,order_metric_cycle_time,diferenca_minutos,diferenca_horas
8,68410055,1/1/2021 2:32:51 AM,1/2/2021 6:57:34 PM,2424.72,2424.716667,40.411944
17,68412721,1/1/2021 2:12:11 PM,1/1/2021 4:12:36 PM,120.42,120.416667,2.006944
18,68413340,1/1/2021 2:14:51 PM,1/1/2021 3:31:54 PM,77.05,77.050000,1.284167
19,68414018,1/1/2021 2:17:31 PM,1/1/2021 3:09:20 PM,51.82,51.816667,0.863611
20,68414309,1/1/2021 2:21:02 PM,1/1/2021 2:50:20 PM,29.30,29.300000,0.488333
21,68414512,1/1/2021 2:24:01 PM,1/1/2021 3:49:47 PM,85.77,85.766667,1.429444
22,68414563,1/1/2021 2:24:26 PM,1/1/2021 3:30:56 PM,66.50,66.500000,1.108333
24,68415103,1/1/2021 2:31:33 PM,1/1/2021 3:38:55 PM,67.37,67.366667,1.122778
25,68415140,1/1/2021 2:31:52 PM,1/1/2021 4:06:40 PM,94.82,94.800000,1.580000
26,68415344,1/1/2021 2:33:32 PM,1/1/2021 4:18:52 PM,105.35,105.333333,1.755556


In [16]:
colunas_investigacao = [
    "order_id",
    "order_status",
    "order_moment_created",
    "order_moment_accepted",
    "order_moment_ready",
    "order_moment_collected",
    "order_moment_delivering",
    "order_moment_delivered",
    "order_moment_finished",
    "order_metric_cycle_time"
]

pedidos.loc[
    pedidos["order_id"] == 68410055,
    colunas_investigacao
].T

,8
order_id,68410055
order_status,FINISHED
order_moment_created,1/1/2021 2:32:51 AM
order_moment_accepted,1/1/2021 2:33:00 AM
order_moment_ready,1/2/2021 6:24:06 PM
order_moment_collected,1/2/2021 6:30:44 PM
order_moment_delivering,1/2/2021 6:35:49 PM
order_moment_delivered,NaN
order_moment_finished,1/2/2021 6:57:34 PM
order_metric_cycle_time,2424.72


In [17]:
validacao = pedidos_finalizados[
    [
        "order_moment_created",
        "order_moment_finished",
        "order_metric_cycle_time"
    ]
].dropna().copy()

inicio = pd.to_datetime(validacao["order_moment_created"])
fim = pd.to_datetime(validacao["order_moment_finished"])

validacao["tempo_calculado"] = (
    (fim - inicio).dt.total_seconds() / 60
)

validacao["diferenca_absoluta"] = (
    validacao["order_metric_cycle_time"]
    - validacao["tempo_calculado"]
).abs()

print("Registros comparados:", len(validacao))
print(
    "Diferença média absoluta:",
    round(validacao["diferenca_absoluta"].mean(), 4)
)
print(
    "Registros com diferença acima de 1 minuto:",
    (validacao["diferenca_absoluta"] > 1).sum()
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\2927043599.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fim = pd.to_datetime(validacao["order_moment_finished"])


Registros comparados: 351656
Diferença média absoluta: 21.4251
Registros com diferença acima de 1 minuto: 18751


In [18]:
print("Distribuição das diferenças:")
print(validacao["diferenca_absoluta"].describe())

print("\nMaiores divergências:")
print(
    validacao.nlargest(
        5, "diferenca_absoluta"
    )[
        [
            "order_metric_cycle_time",
            "tempo_calculado",
            "diferenca_absoluta"
        ]
    ]
)

Distribuição das diferenças:
count    351656.000000
mean         21.425131
std         608.924784
min           0.000000
25%           0.003333
50%           0.003333
75%           0.013333
max      151496.583333
Name: diferenca_absoluta, dtype: float64

Maiores divergências:
       order_metric_cycle_time  tempo_calculado  diferenca_absoluta
45508                   140.55    151637.133333       151496.583333
98068                  1307.38    121533.666667       120226.286667
98300                  1260.42    121486.333333       120225.913333
98624                  1104.42    121330.150000       120225.730000
59429                    32.68     83118.083333        83085.403333


In [19]:
colunas_investigacao = [
    "order_id",
    "order_status",
    "order_moment_created",
    "order_moment_accepted",
    "order_moment_ready",
    "order_moment_collected",
    "order_moment_delivering",
    "order_moment_delivered",
    "order_moment_finished",
    "order_metric_cycle_time"
]

pedidos.loc[45508, colunas_investigacao]

order_id                               72615663
order_status                           FINISHED
order_moment_created       1/20/2021 6:13:08 PM
order_moment_accepted                       NaN
order_moment_ready                          NaN
order_moment_collected                      NaN
order_moment_delivering    1/20/2021 8:03:14 PM
order_moment_delivered     1/20/2021 8:33:41 PM
order_moment_finished       5/6/2021 1:30:16 AM
order_metric_cycle_time                  140.55
Name: 45508, dtype: object

In [20]:
validacao_entrega = pedidos_finalizados[
    [
        "order_moment_created",
        "order_moment_delivered",
        "order_metric_cycle_time"
    ]
].dropna().copy()

criacao = pd.to_datetime(
    validacao_entrega["order_moment_created"]
)

entrega = pd.to_datetime(
    validacao_entrega["order_moment_delivered"]
)

validacao_entrega["tempo_ate_entrega"] = (
    (entrega - criacao).dt.total_seconds() / 60
)

validacao_entrega["diferenca_absoluta"] = (
    validacao_entrega["order_metric_cycle_time"]
    - validacao_entrega["tempo_ate_entrega"]
).abs()

print("Registros comparados:", len(validacao_entrega))

print(
    "Diferença mediana:",
    round(validacao_entrega["diferenca_absoluta"].median(), 4)
)

print(
    "Registros com diferença acima de 1 minuto:",
    (validacao_entrega["diferenca_absoluta"] > 1).sum()
)

C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\3445740464.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  criacao = pd.to_datetime(
C:\Users\Dell\AppData\Local\Temp\ipykernel_17552\3445740464.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  entrega = pd.to_datetime(


Registros comparados: 19003
Diferença mediana: 0.0033
Registros com diferença acima de 1 minuto: 0


### Validação do tempo de ciclo dos pedidos

Foi realizada uma comparação entre o indicador `order_metric_cycle_time`
e os intervalos calculados a partir dos registros de data e hora.

**Comparação entre criação e finalização:**
- Registros analisados: 351.656
- Diferença média absoluta: 21,43 minutos
- Diferença mediana absoluta: aproximadamente 0,0033 minuto
- Registros com divergência superior a 1 minuto: 18.751 (5,33%)

**Comparação entre criação e entrega:**
- Registros analisados: 19.003
- Diferença mediana absoluta: aproximadamente 0,0033 minuto
- Registros com divergência superior a 1 minuto: 0

**Conclusões:**

1. O indicador `order_metric_cycle_time` utiliza minutos.
2. Nos 19.003 pedidos finalizados com data de entrega disponível,
   o indicador coincide com o intervalo entre criação e entrega,
   considerando uma tolerância de 1 minuto.
3. A data de finalização nem sempre representa o momento da entrega.
4. A regra de cálculo para pedidos sem data de entrega ainda precisa
   ser investigada.
5. Os registros originais serão preservados, evitando correções
   ou exclusões sem justificativa.

**Implicação para o projeto:**

O AI Customer Intelligence deverá considerar a disponibilidade e
a consistência dos eventos antes de utilizar indicadores temporais
para avaliar o desempenho operacional dos estabelecimentos.